#### 2.3.1. Загрузка данных и первичный анализ

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    StackingClassifier
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    RocCurveDisplay,
    PrecisionRecallDisplay
)
from sklearn.inspection import permutation_importance

df = pd.read_csv("S06-hw-dataset-01.csv")

df.head()

df.info()

df.describe()

df["target"].value_counts(normalize=True)

df.isna().sum()

df.dtypes

X = df.drop(columns=["target", "id"], errors="ignore")
y = df["target"]


#### 2.3.2. Train/Test-сплит и воспроизводимость

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


#### 2.3.3. Baseline’ы

In [ ]:
dummy = DummyClassifier(strategy="most_frequent", random_state=42)
dummy.fit(X_train, y_train)

y_pred_dummy = dummy.predict(X_test)

print("Dummy accuracy:", accuracy_score(y_test, y_pred_dummy))
print("Dummy f1:", f1_score(y_test, y_pred_dummy, average="macro"))

logreg_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

logreg_pipe.fit(X_train, y_train)

y_pred_lr = logreg_pipe.predict(X_test)
y_proba_lr = logreg_pipe.predict_proba(X_test)

print("LogReg accuracy:", accuracy_score(y_test, y_pred_lr))
print("LogReg f1:", f1_score(y_test, y_pred_lr, average="macro"))

# ROC-AUC только для бинарной задачи
if len(np.unique(y)) == 2:
    print("LogReg ROC-AUC:", roc_auc_score(y_test, y_proba_lr[:, 1]))


#### 2.3.4. Модели недели 6 

In [ ]:
dt = DecisionTreeClassifier(random_state=42)

dt_params = {
    "max_depth": [3, 5, 10, None],
    "min_samples_leaf": [1, 5, 10]
}

dt_cv = GridSearchCV(
    dt,
    dt_params,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1
)

dt_cv.fit(X_train, y_train)
dt_best = dt_cv.best_estimator_

rf = RandomForestClassifier(random_state=42)

rf_params = {
    "n_estimators": [200],
    "max_depth": [None, 10, 20],
    "min_samples_leaf": [1, 5],
    "max_features": ["sqrt", "log2"]
}

rf_cv = GridSearchCV(
    rf,
    rf_params,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1
)

rf_cv.fit(X_train, y_train)
rf_best = rf_cv.best_estimator_

gb = GradientBoostingClassifier(random_state=42)

gb_params = {
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1],
    "max_depth": [3, 5]
}

gb_cv = GridSearchCV(
    gb,
    gb_params,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1
)

gb_cv.fit(X_train, y_train)
gb_best = gb_cv.best_estimator_

stack = StackingClassifier(
    estimators=[
        ("lr", logreg_pipe),
        ("rf", rf_best),
        ("gb", gb_best)
    ],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5,
    n_jobs=-1
)

stack.fit(X_train, y_train)


#### 2.3.5. Метрики качества 

In [ ]:
model = rf_best

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred, average="macro"))

if len(np.unique(y)) == 2:
    y_proba = model.predict_proba(X_test)[:, 1]
    print("ROC-AUC:", roc_auc_score(y_test, y_proba))

    RocCurveDisplay.from_estimator(model, X_test, y_test)
    plt.show()

confusion_matrix(y_test, y_pred)


#### 2.3.6. Интерпретация 

In [ ]:
perm = permutation_importance(
    model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42,
    scoring="f1_macro"
)

importances = pd.Series(
    perm.importances_mean,
    index=X.columns
).sort_values(ascending=False)

importances.head(10)

importances.head(10).plot(kind="barh")
plt.gca().invert_yaxis()
plt.title("Top-10 Permutation Importances")
plt.show()
